# Vidal Case 2 + Active Learning: minimal confirmation
Runs full data, 10% random, and 10% Regression-CLUE for one transfer pair.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this after the new repository is pushed to GitHub.
ACTIVE_REPO_URL = 'https://github.com/zhangcastle/battery-soc-active-learning.git'
!git clone --depth 1 $ACTIVE_REPO_URL /content/battery-soc-active-learning
%pip install -q -e /content/battery-soc-active-learning

In [ ]:
from pathlib import Path

REPO = Path('/content/battery-soc-active-learning')
DRIVE = Path('/content/drive/MyDrive/battery_soc_benchmark')
INPUT = DRIVE / 'data/labelled/vidal_sequence_v2'
CASE1 = DRIVE / 'outputs/reproductions/vidal_2019/case1'
OUTPUT = DRIVE / 'outputs/active_learning_confirmation'
SOURCE, TARGET = 'lg_hg2', 'samsung_30t'

checkpoints = sorted(CASE1.glob(f'{SOURCE}/seed_*/*/checkpoints/epoch_515.pt'))
if len(checkpoints) < 2:
    raise RuntimeError(f'Need two formal Case 1 checkpoints for {SOURCE}; found {len(checkpoints)}')
SOURCE_CHECKPOINT, COMMITTEE_CHECKPOINT = checkpoints[-2:]
print(SOURCE_CHECKPOINT)
print(COMMITTEE_CHECKPOINT)

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, str(REPO / 'scripts/run_active_learning_confirmation.py'),
    '--source', SOURCE, '--target', TARGET,
    '--config', str(REPO / 'configs/reproductions/vidal_2019/vidal_case2_diagnostic.yaml'),
    '--input-root', str(INPUT), '--output-root', str(OUTPUT),
    '--source-checkpoint', str(SOURCE_CHECKPOINT),
    '--committee-checkpoint', str(COMMITTEE_CHECKPOINT),
    '--budget', '0.10', '--run-mode', 'smoke', '--device', 'auto',
], cwd=REPO, check=True)

In [ ]:
import pandas as pd
pd.read_csv(OUTPUT / 'confirmation_summary.csv')